# Presentacion: Topologia AS de Chile (BGP vs RIPE vs Combinado)

Version reducida para slides:
- Menos tablas extensas
- Mas visualizaciones comparativas
- Mensajes clave al cierre


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

DATA_DIR = Path('../data/csv') if Path('../data/csv').exists() else Path('data/csv')


def load_dataset(name):
    n = pd.read_csv(DATA_DIR / name / 'nodes.csv')
    e = pd.read_csv(DATA_DIR / name / 'edges.csv')
    n['name'] = n['name'].fillna('')
    n['degree'] = n['in_degree'] + n['out_degree']
    return n, e


def asn_set(nodes):
    return set(nodes['asn'].astype(int))


def edge_set(nodes, edges):
    id2asn = dict(zip(nodes['node_id'], nodes['asn']))
    return {(int(id2asn[s]), int(id2asn[d])) for s, d in edges[['src_id', 'dst_id']].itertuples(index=False)}


def jaccard(a, b):
    u = a | b
    return len(a & b) / len(u) if u else 0.0


def normalize(s):
    return s / max(float(s.max()), 1.0)

bgp_nodes, bgp_edges = load_dataset('bgp')
ripe_nodes, ripe_edges = load_dataset('ripe_atlas')
merged_nodes, merged_edges = load_dataset('merged')

asn_b = asn_set(bgp_nodes)
asn_r = asn_set(ripe_nodes)
edge_b = edge_set(bgp_nodes, bgp_edges)
edge_r = edge_set(ripe_nodes, ripe_edges)

# Chile seed (presentacion)
chile_explicit = set(merged_nodes.loc[merged_nodes['name'].str.contains(r'\bchile\b', case=False, regex=True), 'asn'].astype(int))
chile_manual = {27986, 6568, 27925, 27651, 22047, 52341, 18822, 14117, 10834, 20015, 6471, 6429, 14259, 27678, 20191, 23140, 64112}
chile_asns = chile_explicit | chile_manual

print('Cargado:', len(bgp_nodes), len(ripe_nodes), len(merged_nodes), 'nodos')

Cargado: 16988 127 9839 nodos


## Slide 1: Cobertura por fuente

In [2]:
cov = pd.DataFrame({
    'topologia': ['BGP', 'RIPE Atlas', 'Combinado'],
    'nodos': [len(bgp_nodes), len(ripe_nodes), len(merged_nodes)],
    'aristas': [len(bgp_edges), len(ripe_edges), len(merged_edges)],
})

fig = make_subplots(rows=1, cols=2, subplot_titles=('Nodos', 'Aristas'))
fig.add_trace(go.Bar(x=cov['topologia'], y=cov['nodos'], text=cov['nodos'], textposition='outside', name='Nodos'), row=1, col=1)
fig.add_trace(go.Bar(x=cov['topologia'], y=cov['aristas'], text=cov['aristas'], textposition='outside', name='Aristas'), row=1, col=2)
fig.update_layout(title='Cobertura estructural por fuente', showlegend=False)
fig.show()

## Slide 2: Solapamiento BGP vs RIPE

In [3]:
overlap_df = pd.DataFrame([
    {'tipo': 'Nodos', 'grupo': 'Solo BGP', 'cantidad': len(asn_b - asn_r)},
    {'tipo': 'Nodos', 'grupo': 'Comunes', 'cantidad': len(asn_b & asn_r)},
    {'tipo': 'Nodos', 'grupo': 'Solo RIPE', 'cantidad': len(asn_r - asn_b)},
    {'tipo': 'Aristas', 'grupo': 'Solo BGP', 'cantidad': len(edge_b - edge_r)},
    {'tipo': 'Aristas', 'grupo': 'Comunes', 'cantidad': len(edge_b & edge_r)},
    {'tipo': 'Aristas', 'grupo': 'Solo RIPE', 'cantidad': len(edge_r - edge_b)},
])

fig = px.bar(overlap_df, x='grupo', y='cantidad', color='tipo', barmode='group', text='cantidad',
             title='Solapamiento BGP vs RIPE')
fig.update_traces(textposition='outside')
fig.show()

print('Jaccard nodos:', round(jaccard(asn_b, asn_r), 4), '| Jaccard aristas:', round(jaccard(edge_b, edge_r), 6))

Jaccard nodos: 0.0072 | Jaccard aristas: 0.000188


## Slide 3: Distribucion de grado

In [4]:
deg_df = pd.concat([
    bgp_nodes[['degree']].assign(topologia='BGP'),
    ripe_nodes[['degree']].assign(topologia='RIPE Atlas'),
    merged_nodes[['degree']].assign(topologia='Combinado'),
], ignore_index=True)

fig = px.histogram(deg_df, x='degree', color='topologia', barmode='overlay', opacity=0.65,
                   marginal='box', log_y=True, title='Distribucion de grado (escala log en Y)')
fig.show()

## Slide 4: Top ASNs chilenos por relevancia

In [5]:
cl = merged_nodes[merged_nodes['asn'].isin(chile_asns)].copy()
cl['score'] = 0.5 * normalize(cl['degree']) + 0.5 * normalize(cl['path_occurrences'])
top = cl.sort_values(['score', 'degree', 'path_occurrences'], ascending=False).head(10).copy()
top['label'] = top['asn'].astype(str) + ' - ' + top['name'].replace('', '(sin nombre)')

fig = px.bar(top.sort_values('score'), y='label', x='score', orientation='h', text='score',
             title='Top 10 ASNs chilenos (score combinado)')
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.show()

## Slide 5: ASNs comunes con nombre de organizacion

In [6]:
common = sorted(asn_b & asn_r)
cb = pd.DataFrame({'asn': common})
cb['deg_bgp'] = cb['asn'].map(dict(zip(bgp_nodes['asn'], bgp_nodes['degree'])))
cb['deg_ripe'] = cb['asn'].map(dict(zip(ripe_nodes['asn'], ripe_nodes['degree'])))
cb['org_bgp'] = cb['asn'].map(dict(zip(bgp_nodes['asn'], bgp_nodes['name'].fillna('').astype(str))))
cb['org_ripe'] = cb['asn'].map(dict(zip(ripe_nodes['asn'], ripe_nodes['name'].fillna('').astype(str))))
cb['organizacion'] = cb['org_ripe'].replace('', np.nan).fillna(cb['org_bgp'].replace('', np.nan)).fillna('(sin nombre)')

display(cb.sort_values(['deg_ripe', 'deg_bgp'], ascending=False).head(12)[['asn', 'organizacion', 'deg_bgp', 'deg_ripe']])

,asn,organizacion,deg_bgp,deg_ripe
31,14259,GTD Chile,156,42
12,6429,CLARO CHILE AS6429,31,34
29,13335,Cloudflare,1631,30
16,6939,Hurricane Electric,10225,29
17,7004,CTC Transmisiones Regionales S.A.,60,25
69,52304,(sin nombre),6,25
67,52234,(sin nombre),11,24
9,3549,Lumen AS 3549,145,20
8,3356,Lumen AS3356,2525,19
33,15208,(sin nombre),4,18


## Slide 6: Mensajes clave

In [7]:
pearson = cb['deg_bgp'].corr(cb['deg_ripe']) if len(cb) > 1 else np.nan
msg = [
    f"- Cobertura: BGP={len(bgp_nodes):,} nodos, RIPE={len(ripe_nodes):,}, Combinado={len(merged_nodes):,}.",
    f"- Solapamiento de nodos BGP∩RIPE: {len(asn_b & asn_r):,} (Jaccard={jaccard(asn_b, asn_r):.4f}).",
    f"- Solapamiento de aristas BGP∩RIPE: {len(edge_b & edge_r):,} (Jaccard={jaccard(edge_b, edge_r):.6f}).",
    f"- Correlacion de grado en ASNs comunes (Pearson): {pearson:.3f}.",
    '- Lectura: BGP aporta amplitud, RIPE aporta evidencia operacional, combinado mejora la vista global.'
]
display(Markdown("\\n".join(msg)))

- Cobertura: BGP=16,988 nodos, RIPE=127, Combinado=9,839.\n- Solapamiento de nodos BGP∩RIPE: 122 (Jaccard=0.0072).\n- Solapamiento de aristas BGP∩RIPE: 90 (Jaccard=0.000188).\n- Correlacion de grado en ASNs comunes (Pearson): 0.221.\n- Lectura: BGP aporta amplitud, RIPE aporta evidencia operacional, combinado mejora la vista global.